# Security, Roles, and Access Control

In a real-world enterprise, a database is not a free-for-all sandbox. Hundreds of people and applications connect to the same database every day. If a Junior Analyst accidentally typed `DROP TABLE Customers;`, the entire company could go bankrupt.

To prevent this, databases use a strict system of **Users**, **Roles**, and **Permissions** to ensure everyone only has access to exactly what they need to do their job, and absolutely nothing more. This is known in the industry as the **Principle of Least Privilege**.

*(Note: Because our trusty SQLite sandbox is designed for single-user local files, it does not actually have built-in user management! For Sections 1 through 3, we will look at standard SQL syntax used in massive systems like PostgreSQL and MySQL. In Section 4, we will jump back into Python/SQLite to show a clever Data Science security trick!)*

---


# 1. Authentication vs. Authorization
Database security is broken down into two distinct concepts:

* **Authentication (Who are you?)**: This is the login screen. You provide a username and a password. The database checks its records and says, *"Okay, I believe you are Alice."*
* **Authorization (What are you allowed to do?)**: Once Alice is inside, the database checks her ID badge. *"Alice is allowed to READ the Sales table, but she is NOT allowed to DELETE anything."*

# 2. Creating Users and Roles
Instead of giving permissions to every single user individually, administrators create **Roles** (like a job title), give permissions to the *Role*, and then assign Users to that Role.

```sql
-- 1. Create a general Role for the Data Science team
CREATE ROLE data_scientist;

-- 2. Create specific Users with passwords
CREATE USER 'alice'@'localhost' IDENTIFIED BY 'SecurePass123!';
CREATE USER 'bob_intern'@'localhost' IDENTIFIED BY 'InternPass456!';

-- 3. Assign Alice to the Data Scientist role
GRANT data_scientist TO 'alice'@'localhost';
```

# 3. GRANT and REVOKE (Managing Permissions)

The `GRANT` command is how you give permissions, and `REVOKE` is how you take them away.

### Common Permissions:

* **`SELECT`**: Allowed to read data. (This is what Data Scientists usually ask for!)
* **`INSERT` / `UPDATE`**: Allowed to add or modify data.
* **`DELETE` / `DROP`**: Allowed to remove data or destroy tables. (Very dangerous!)
* **`ALL PRIVILEGES`**: Full administrative access. Typically reserved for database administrators (DBAs).

```sql
-- Give the Data Science role permission to READ the Sales table
GRANT SELECT ON Sales TO data_scientist;

-- Give Alice permission to create new tables for her machine learning models
GRANT CREATE TABLE ON ML_Predictions TO 'alice'@'localhost';

-- Uh oh, Bob the intern accidentally updated a live record. 
-- Let's take away his UPDATE privileges immediately!
REVOKE UPDATE ON Customers FROM 'bob_intern'@'localhost';
```

# 4. Using "Views" to Mask Sensitive Data
What if a Data Scientist needs to analyze the `Employees` table to predict employee turnover, but the table contains highly sensitive `Salary` and `Social_Security_Number` columns? You can't `GRANT SELECT` on the whole table!

The solution is a **View**. A View is a "virtual table." It runs a saved SQL query and presents the results as if it were a real table. You can use a View to hide sensitive columns, and then grant the Data Scientist access *only* to the View.

Let's jump back into Python and SQLite to see this in action!

In [1]:
import sqlite3
import pandas as pd

# Connect to our sandbox
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# 1. Create the real, highly sensitive HR table
cursor.executescript("""
CREATE TABLE HR_Employees (
    emp_id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    social_security TEXT,
    salary DECIMAL(10, 2)
);

INSERT INTO HR_Employees VALUES 
(1, 'Alice', 'Engineering', '123-45-678', 120000),
(2, 'Bob', 'Sales', '987-65-432', 85000);
""")

print("--- The Real Table (Highly Restricted) ---")
display(pd.read_sql_query("SELECT * FROM HR_Employees;", conn))

# 2. Create a secure VIEW that hides the Social Security and Salary columns
cursor.executescript("""
CREATE VIEW Safe_Employee_Data AS
SELECT 
    emp_id, 
    name, 
    department 
FROM HR_Employees;
""")

# 3. The Data Scientist now queries the VIEW, completely unaware of the hidden data!
print("\n--- The Secure View (What the Data Scientist sees) ---")
display(pd.read_sql_query("SELECT * FROM Safe_Employee_Data;", conn))

conn.close()

--- The Real Table (Highly Restricted) ---


,emp_id,name,department,social_security,salary
0,1,Alice,Engineering,123-45-678,120000
1,2,Bob,Sales,987-65-432,85000



--- The Secure View (What the Data Scientist sees) ---


,emp_id,name,department
0,1,Alice,Engineering
1,2,Bob,Sales


# 5. Row-Level Security (RLS)
Sometimes, hiding *columns* isn't enough; you need to hide specific *rows*. 

For example, imagine a hospital database. Doctor Smith should only be allowed to `SELECT` rows where `treating_physician = 'Dr. Smith'`. She should not be able to read the medical records of Doctor Patel's patients. 

Modern databases (like PostgreSQL) use **Row-Level Security (RLS)** to automatically filter the results of a query based on the specific user who is logged in, ensuring absolute privacy.

---

## Real-World Use Case or Analogy:
Think of Database Security like working at a **Highly Secure Corporate Office**:

* **Authentication (`CREATE USER`)**: The security guard at the front desk. You show your ID and give your password. The guard gives you a blank physical keycard and lets you into the lobby. 
* **Roles (`CREATE ROLE`)**: You are handed a red lanyard that says "Data Team". The janitors get blue lanyards, and the CEO gets a gold lanyard.
* **Permissions (`GRANT / REVOKE`)**: The security system programs your specific keycard. 
    * `GRANT SELECT`: Your card opens the door to the library so you can read books.
    * `GRANT INSERT`: Your card opens the door to the mailroom so you can drop off packages.
    * `REVOKE DELETE`: If you get caught throwing away important files, security remotely disables your card's access to the incinerator room.
* **Views (`CREATE VIEW`)**: You need to look at a classified financial document to do your math. Instead of giving you the original document, a manager takes a black sharpie, censors all the personal names and bank account numbers, photocopies it, and gives you the safe copy.

---